# Earthquake Vision — Malatya hasar segmentasyonu: Colab eğitimi

**Önemli:** CodeOcean dosya sunucusu Colab'dan gelen indirmeleri engelliyor (HTTP 403). Bu yüzden veri **önce kendi bilgisayarınızda** indirilir ve Google Drive üzerinden buraya getirilir.

### Bilgisayarınızda (bir kez)
```bash
cd ~/earthquake-vision && source venv/bin/activate && git pull
python -m src.training.audit_dataset                                  # PASSED olmalı
python -m src.training.prepare_malatya --variant malatya
python -m src.training.prepare_malatya --variant kate7_eval_malatya   # ~400 MB
cd data/raw && zip -qr ~/Desktop/kate_cd_data.zip kate_cd && cd ../..
```
Sonra `kate_cd_data.zip` dosyasını Google Drive'ınızın ana klasörüne (**Drive'ım**) yükleyin.

### Bu not defterinde
1. GPU kontrolü
2. Repoyu klonlama (private repo için token)
3. Paketler
4. Veriyi Drive'dan alma
5. **Denetim kapısı:** repodaki denetim raporu `PASSED` olmalı ve Drive'dan gelen metadata, denetlenen dosyalarla birebir aynı olmalı (SHA-256)
6. YOLO formatına dönüştürme (iki varyant)
7. Deney A (yalnızca Malatya) ve Deney B (7 şehirle eğit, yalnızca Malatya'da ölç)
8. Değerlendirme, karşılaştırma tablosu, paneller, sonuçları indirme

Her adım hata olursa durur; bir hücre kırmızı hata verirse sonrakine geçmeyin.

**Başlamadan önce:** *Runtime → Change runtime type → T4 GPU* (veya L4/A100).

## 1. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU YOK: Runtime -> Change runtime type -> GPU seçin"

## 2. Repoyu klonla

GitHub → *Settings → Developer settings → Personal access tokens → Fine-grained tokens* → yalnızca bu repo için **Contents: Read-only**. Token gizli sorulur ve not defterine kaydedilmez.

In [ ]:
import getpass, os, subprocess
REPO = "selimhancil/earthquake-damage-segmentation"
os.chdir("/content")
if not os.path.isdir("earthquake-damage-segmentation"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git"], check=True)
    del token
else:
    subprocess.run(["git", "-C", "earthquake-damage-segmentation", "pull", "--ff-only"], check=False)
os.chdir("/content/earthquake-damage-segmentation")
!git log --oneline -1

def run(cmd):
    """Komutu çalıştırır; başarısız olursa hücreyi durdurur (sessizce devam etmez)."""
    print("$", cmd)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise SystemExit(f"Komut başarısız (çıkış kodu {result.returncode}): {cmd}")

## 3. Paketler
Colab'da PyTorch hazır gelir; yalnızca Ultralytics kurulur.

In [ ]:
run("pip install -q ultralytics")
import ultralytics, torch, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 4. Veriyi Google Drive'dan al

Drive bağlantı izni istenir. `kate_cd_data.zip` dosyası *Drive'ım* klasöründe olmalı (başka yerdeyse `ZIP` yolunu değiştirin).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/kate_cd_data.zip"
assert os.path.isfile(ZIP), f"{ZIP} bulunamadı. Bilgisayarınızda oluşturup Drive'ım klasörüne yükleyin."
os.makedirs("data/raw", exist_ok=True)
run(f'unzip -q -o "{ZIP}" -d data/raw')
for variant in ("malatya", "kate7_eval_malatya"):
    path = f"data/raw/kate_cd/selection_{variant}.json"
    print(variant, "hazır" if os.path.isfile(path) else "EKSİK (bilgisayarda prepare çalıştırılmamış)")

## 5. Denetim kapısı

Denetim bilgisayarınızda çalıştırıldı ve raporu repoya eklendi (`reports/data_audit.json`). Burada:
- raporun `PASSED` olduğu,
- Drive'dan gelen metadata dosyalarının denetlenen dosyalarla **aynı** olduğu (SHA-256)
kontrol edilir. Veri yeniden indirilmez (`--offline`); eksik dosya varsa durur.

In [ ]:
import json
audit = json.load(open("reports/data_audit.json"))
assert audit["status"] == "PASSED", "Denetim geçmemiş; eğitim yapılmayacak: " + str(audit["blockers"])
print("Denetim:", audit["status"], "| Malatya:", audit["malatya"]["by_split"])
run("python -m src.training.prepare_malatya --variant malatya --offline")
run("python -m src.training.prepare_malatya --variant kate7_eval_malatya --offline")

## 6. YOLO formatına dönüştürme

In [ ]:
run("python -m src.training.convert_to_yolo --variant malatya")
run("python -m src.training.convert_to_yolo --variant kate7_eval_malatya")

## 7A. Deney A — yalnızca Malatya

Başlangıç ayarları: `imgsz=512, epochs=30, batch=8, patience=10, seed=42` (performans garantisi değildir).

In [ ]:
run("python -m src.training.train --variant malatya --name A_malatya_only --device 0")

## 7B. Deney B — 7 şehirle eğit, yalnızca Malatya'da ölç

In [ ]:
run("python -m src.training.train --variant kate7_eval_malatya --name B_kate7_eval_malatya --device 0")

## 8. Değerlendirme (yalnızca Malatya test kümesi)

- mAP: `conf=0.001` (arayüzdeki 0.70 eşiği burada kullanılmaz).
- Piksel IoU/Dice: `conf ≥ 0.25 / 0.50 / 0.70`, bootstrap %95 güven aralığıyla.
- Test kümesi 4 görüntü: sonuçları kesin değil, belirsizlik aralığıyla yorumlayın.

In [ ]:
run("python -m src.training.evaluate --weights weights/A_malatya_only_best.pt --variant malatya --name A_malatya_only --device 0")
run("python -m src.training.evaluate --weights weights/B_kate7_eval_malatya_best.pt --variant kate7_eval_malatya --name B_kate7_eval_malatya --device 0")

## 9. Karşılaştırma tablosu

In [ ]:
rows = []
for name in ["A_malatya_only", "B_kate7_eval_malatya"]:
    r = json.load(open(f"reports/evaluation_{name}.json"))
    s = r.get("ultralytics_seg", {})
    m = r["pixel_metrics_by_conf_threshold"]["0.50"]
    rows.append((name, r["n_images"], s.get("mask_map50"), s.get("mask_map50_95"),
                 m["micro_iou"], m["micro_iou_ci95"], m["micro_dice_f1"], m["area"]["relative_area_error"]))
print(f"{'deney':24} {'n':>2} {'mAP50':>7} {'mAP50-95':>9} {'IoU@0.5':>8} {'IoU %95 GA':>16} {'Dice@0.5':>9} {'alan hatası':>11}")
for r in rows:
    print(f"{r[0]:24} {r[1]:>2} {str(r[2]):>7} {str(r[3]):>9} {str(r[4]):>8} {str(r[5]):>16} {str(r[6]):>9} {str(r[7]):>11}")

## 10. Örnek paneller (öncesi | sonrası | ground truth | tahmin)

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob("reports/figures/*/*.png"))[:8]:
    print(path); display(Image(path))

## 11. Sonuçları indir

Uygulamada kullanacağınız modeli `weights/best_model.pt` olarak bilgisayarınızdaki projeye koyun. `reports/` klasörü makale için kanıt niteliğindedir.

In [ ]:
run("zip -q -r /content/earthquake_vision_results.zip weights/*_best.pt reports/ runs/segment/*/results.csv runs/segment/*/args.yaml")
from google.colab import files
files.download("/content/earthquake_vision_results.zip")